# SEMSYNTH Kaggle: reproducible experiments
Enable Internet for GitHub clone; GPU is not required. Set REPO_URL, or attach the complete full source ZIP. Run the cell below.
Offline mode needs no API key and does not measure real LLM quality.


In [ ]:
# SEMSYNTH Kaggle — ONE CELL. CPU is sufficient; Internet ON for cloning.
# Upload the current complete project to GitHub, then set REPO_URL below.
# Alternatively attach a FULL project source ZIP to this notebook's /kaggle/input.
import os, sys, subprocess, shutil, zipfile, glob
from pathlib import Path
REPO_URL = ""  # e.g. https://github.com/your-account/your-repository.git
BRANCH = "main"
MODE = "offline"  # offline=32 keyless controlled cases; real=live model generations
PROVIDER = "gemini"
MODEL = ""    # blank uses the configured provider default
QUICK = False  # True keeps one task per family (8 tasks)
MAX_TASKS = None  # Set to 1 for a real-API smoke check
METHODS = "direct1,first_valid,balanced"
NUM_SHARDS, SHARD_INDEX = 1, 0  # set (2,0) and (2,1) in two notebooks
MAX_MINUTES = 600  # leave buffer for Kaggle ZIP/export
TOP_K, MAX_QUESTIONS, SEED = 3, 3, 3101
PROJECT = Path('/kaggle/working/semsynth_repo')

if REPO_URL.strip():
    if not (PROJECT / 'run_experiments.py').exists():
        subprocess.run(['git','clone','--depth','1','--branch',BRANCH,REPO_URL,str(PROJECT)],check=True)
    else:
        subprocess.run(['git','-C',str(PROJECT),'fetch','--depth','1','origin',BRANCH],check=True)
        subprocess.run(['git','-C',str(PROJECT),'reset','--hard','FETCH_HEAD'],check=True)
        print('Updated existing clone:', PROJECT)
else:
    archives=sorted(Path('/kaggle/input').glob('**/*.zip'))
    if not archives:
        raise ValueError('Set REPO_URL to your current GitHub repository, or attach the FULL project ZIP as a Kaggle input.')
    PROJECT.mkdir(parents=True,exist_ok=True)
    with zipfile.ZipFile(archives[0]) as z:
        if any(n.startswith('/') or '..' in Path(n).parts for n in z.namelist()):
            raise ValueError('Unsafe ZIP member')
        z.extractall(PROJECT)
    matches=list(PROJECT.rglob('run_experiments.py'))
    if len(matches)!=1:
        raise RuntimeError('The uploaded project ZIP must include run_experiments.py. Upload the complete source project.')
    PROJECT=matches[0].parent
print('Working repository:',PROJECT)
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(PROJECT/'requirements-kaggle.txt')],check=True)
if MODE=='real':
    if not MODEL.strip(): raise ValueError("Set MODEL to a model verified for your account")
    subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(PROJECT/'requirements-llm.txt')],check=True)
    from kaggle_secrets import UserSecretsClient
    key={'gemini':'GEMINI_API_KEY','openai':'OPENAI_API_KEY','openrouter':'OPENROUTER_API_KEY'}[PROVIDER]
    secret=UserSecretsClient().get_secret(key)
    if not secret: raise RuntimeError(f'Please configure Kaggle secret {key}')
    os.environ[key]=secret

output=Path('/kaggle/working/semsynth_results')
commands=[
 [sys.executable,'-m','pytest','-q'],
 [sys.executable,'scripts/preflight.py','--mode',MODE,'--provider',PROVIDER,'--output-dir',str(output)],
]
for command in commands:
    subprocess.run(command,cwd=str(PROJECT),check=True)
args=[sys.executable,'run_experiments.py','--mode',MODE,'--provider',PROVIDER,
      '--top-k',str(TOP_K),'--max-questions',str(MAX_QUESTIONS),'--seed',str(SEED),'--methods',METHODS,
      '--num-shards',str(NUM_SHARDS),'--shard-index',str(SHARD_INDEX),
      '--max-minutes',str(MAX_MINUTES),'--output-dir',str(output)]
if QUICK: args.append('--quick')
if MAX_TASKS is not None: args.extend(['--max-tasks',str(MAX_TASKS)])
args.append('--require-complete')
if MODEL: args.extend(['--model',MODEL])
subprocess.run(args,cwd=str(PROJECT),check=True)
print('Results archives:',[str(p) for p in output.rglob('SEMSYNTH_RESULTS.zip')])
